[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/visualizacion_datos/01_exploracion/exploracion_hallazgos.ipynb)

# Fase 1: Exploración y Hallazgos
## Detección de Deslizamientos de Tierra con Machine Learning
**Asignatura:** Visualización de Datos · 2026  
**Dataset:** Landslide4Sense — Sentinel-1/2 · ALOS DEM · 14 canales · 3799 muestras  
**Objetivo:** Descubrir qué modelos y qué señales del terreno son más discriminativas para detectar deslizamientos.

---

In [ ]:
# ── Setup: detectar entorno (Colab vs. local) y configurar rutas ──────────────
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('Landslides_-Applied-ML-Course'):
        os.system('git clone https://github.com/apmontesp/Landslides_-Applied-ML-Course.git')
    DATA_DIR = 'Landslides_-Applied-ML-Course/visualizacion_datos/data'
    FIG_DIR  = 'Landslides_-Applied-ML-Course/visualizacion_datos/data/figures'
else:
    DATA_DIR = '../data'
    FIG_DIR  = '../data/figures'

os.makedirs(FIG_DIR, exist_ok=True)
print(f'Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'DATA_DIR: {DATA_DIR}')

In [ ]:
# ── Importaciones ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Estilo global: fondo blanco, cuadrícula gris tenue, sin marcos innecesarios
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'axes.edgecolor'    : '#D1D5DB',
    'axes.grid'         : True,
    'grid.color'        : '#E5E7EB',
    'grid.linewidth'    : 0.8,
    'font.size'         : 11,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 11,
    'figure.figsize'    : (10, 5),
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

# Paleta del proyecto
ROJO    = '#D62728'
VERDE   = '#2CA02C'
VERDE_C = '#86EFAC'   # verde claro (protocolo alternativo)
PURPURA = '#7C3AED'
PURPURA_C = '#C4B5FD' # púrpura claro
GRIS    = '#9CA3AF'
GRIS_C  = '#E5E7EB'

print('Librerías cargadas. Estilo: fondo blanco, cuadrícula gris tenue.')

In [ ]:
# ── Carga de datos de resultados ──────────────────────────────────────────────
import json as _json

_df_raw = pd.read_csv(f'{DATA_DIR}/comparison_table.csv')
for col in ['F1 medio', 'Std', 'AUC-ROC', 'Precisión', 'Recall', 'IoU']:
    _df_raw[col] = pd.to_numeric(_df_raw[col], errors='coerce')

df_channels = pd.read_csv(f'{DATA_DIR}/channel_stats_by_class.csv')

with open(f'{DATA_DIR}/final_summary.json') as f:
    summary = _json.load(f)

# df_models se sobreescribe en la Exploración 1 tras limpiar AUC; aquí solo preview
print('Datos cargados:')
print(f'  Modelos evaluados : {len(_df_raw)}')
print(f'  Canales analizados: {len(df_channels)}')
print(f'  Mejor modelo      : {summary["best_model"]} (F1={summary["best_f1"]:.4f})')
print()
_df_raw

## 1. Pregunta de Negocio

> **¿Qué modelo de Machine Learning detecta mejor los deslizamientos de tierra en imágenes satelitales multiespectrales, y qué características físicas del terreno son más determinantes para la predicción?**

**Contexto:** Los deslizamientos de tierra causan miles de muertes y miles de millones de dólares en pérdidas anuales. Detectarlos automáticamente desde imágenes satelitales permitiría alertas tempranas y mapeo rápido de riesgo. El dataset Landslide4Sense contiene imágenes de 14 bandas (ópticas, SAR, topográficas) etiquetadas como landslide (positivo) o no-landslide (negativo).

**Modelos evaluados:**
- **Clásicos:** Logistic Regression, SVM (RBF), Random Forest
- **Deep Learning:** ResNet-50, EfficientNet-B4, U-Net ResNet-34

**Métricas clave:** F1 Score (balance precisión-recall), AUC-ROC, Recall (prioridad en alertas tempranas)

---
## 2. Exploración 1: Que modelo tiene mayor F1 Score?

**Hipótesis inicial:** Esperamos que los modelos de Deep Learning superen a los clásicos, dado su mayor capacidad de representación.

Comenzamos con una visualización exploratoria directa: barras simples de F1 medio por modelo.

In [ ]:
# ── Exploración 1: F1 Score por modelo ───────────────────────────────────────
import json as _json, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches

# Datos: dos evaluaciones para modelos clásicos
#   • '14 bandas del satélite'          — protocolo optimizado, todas las señales espectrales
#   • 'características básicas'         — protocolo comparable a literatura (HOG+DEM+NDVI)
f1_14bandas = {'Logistic Reg.':0.7886, 'SVM (RBF)':0.7974, 'Random Forest':0.8368,
               'ResNet-50':0.7865, 'EfficientNet-B4':0.7554, 'U-Net ResNet-34':0.6949}
f1_basico   = {'Logistic Reg.':0.6705, 'SVM (RBF)':0.7328, 'Random Forest':0.7724}

tipos = {'Logistic Reg.':'Clásico','SVM (RBF)':'Clásico','Random Forest':'Clásico',
         'ResNet-50':'Deep Learning','EfficientNet-B4':'Deep Learning','U-Net ResNet-34':'Deep Learning'}
orden  = sorted(f1_14bandas, key=lambda m: f1_14bandas[m], reverse=True)

COL_CL_A = '#2CA02C'   # verde oscuro  — 14 bandas
COL_CL_B = '#86EFAC'   # verde claro   — básico
COL_DL   = '#7C3AED'   # púrpura       — Deep Learning
GRIS     = '#9CA3AF'

x = np.arange(len(orden)); ancho = 0.36
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_facecolor('white')

for i, modelo in enumerate(orden):
    tipo  = tipos[modelo]
    val_a = f1_14bandas[modelo]
    col_a = COL_CL_A if tipo == 'Clásico' else COL_DL

    if modelo in f1_basico:
        ax.bar(x[i]-ancho/2, val_a,             ancho, color=col_a,   zorder=3)
        ax.bar(x[i]+ancho/2, f1_basico[modelo], ancho, color=COL_CL_B, zorder=3)
        ax.text(x[i]-ancho/2, val_a+0.012,
                f'{val_a:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.text(x[i]+ancho/2, f1_basico[modelo]+0.012,
                f'{f1_basico[modelo]:.3f}', ha='center', va='bottom', fontsize=9, color='#4B5563')
    else:
        ax.bar(x[i], val_a, ancho*0.9, color=col_a, zorder=3)
        ax.text(x[i], val_a+0.012, f'{val_a:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(orden, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('F1 Score  (0 = peor  ·  1 = mejor)', fontsize=11)
ax.set_xlabel('Modelo de Machine Learning', fontsize=11)
ax.set_title('F1 Score por Modelo — Detección de Deslizamientos de Tierra', fontsize=13)
ax.set_ylim(0, 1.05)

ax.axhline(0.80, color='#EF4444', linewidth=1.4, linestyle='--', zorder=1)
ax.text(len(orden)-0.45, 0.815, 'F1 = 0.80  (umbral operativo)',
        fontsize=9, color='#EF4444')

# Separador clásico / DL
ax.axvline(2.5, color='#D1D5DB', linewidth=1.2, linestyle=':', zorder=0)
ax.text(1.0, 0.97, 'Modelos Clásicos',  ha='center', fontsize=9, color=GRIS, style='italic')
ax.text(4.0, 0.97, 'Deep Learning',     ha='center', fontsize=9, color=GRIS, style='italic')

leyenda = [
    mpatches.Patch(color=COL_CL_A, label='Clásico — 14 bandas del satélite'),
    mpatches.Patch(color=COL_CL_B, label='Clásico — características básicas del terreno'),
    mpatches.Patch(color=COL_DL,   label='Deep Learning — validación cruzada 5 folds'),
]
ax.legend(handles=leyenda, fontsize=9, loc='lower left', framealpha=0.9)
ax.yaxis.grid(True, color='#E5E7EB', linewidth=0.8, zorder=0)
ax.xaxis.grid(False)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_1_f1_barras.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

import pandas as pd
df_models = pd.read_csv(f'{DATA_DIR}/comparison_table.csv')
for col in ['F1 medio','Std','AUC-ROC','Precisión','Recall','IoU']:
    df_models[col] = pd.to_numeric(df_models[col], errors='coerce')
print('Figura guardada: exploracion_1_f1_barras.png')


### Análisis Exploración 1

**Observacion inesperada:** El modelo **Random Forest (F1=0.837)** supera a todos los modelos de Deep Learning, incluyendo ResNet-50 (F1=0.784) y EfficientNet-B4 (F1=0.755). U-Net ResNet-34 — diseñada específicamente para segmentacion semantica — obtiene el peor resultado (F1=0.444).

**¿Por qué?** Exploramos mas en la siguiente visualizacion...

In [ ]:
# ── Exploración 1b: Trade-off Precisión vs Recall + referencias de literatura ──
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches

datos_pr = [
    {'Modelo':'Logistic Reg.', 'Precisión':0.7971,'Recall':0.7806,'F1':0.7886,'Tipo':'Clásico'},
    {'Modelo':'SVM (RBF)',     'Precisión':0.8193,'Recall':0.7777,'F1':0.7974,'Tipo':'Clásico'},
    {'Modelo':'Random Forest', 'Precisión':0.7439,'Recall':0.9569,'F1':0.8368,'Tipo':'Clásico'},
    {'Modelo':'ResNet-50',     'Precisión':0.7219,'Recall':0.8771,'F1':0.7865,'Tipo':'Deep Learning'},
]
df_pr = pd.DataFrame(datos_pr)

# Referencias de literatura (solo F1 disponible — se muestran como líneas ISO-F1)
literatura = [
    {'autor':'Ghorbanzadeh, 2022', 'f1':0.717, 'tipo':'DL'},
    {'autor':'L4S Competition, 2022','f1':0.739,'tipo':'DL'},
    {'autor':'Multi-scale diff., 2024','f1':0.760,'tipo':'DL'},
    {'autor':'Enhanced U-Net++, 2025','f1':0.841,'tipo':'DL'},
]

COL_CL = '#2CA02C'; COL_DL = '#7C3AED'
GRIS_LIT = '#6B7280'   # gris para líneas de literatura

def dibujar_panel(ax, xlim, ylim, titulo):
    ax.set_facecolor('white')

    # ── Líneas ISO-F1 de literatura (fondo) ──
    prec_arr = np.linspace(0.40, 0.999, 600)
    for ref in literatura:
        f1v = ref['f1']
        rec_arr = (f1v * prec_arr) / (2*prec_arr - f1v + 1e-9)
        mask = (rec_arr > 0.3) & (rec_arr <= 1.0)
        if mask.sum() < 2: continue
        ax.plot(prec_arr[mask], rec_arr[mask],
                color=GRIS_LIT, linewidth=0.9, linestyle='--',
                alpha=0.55, zorder=1)
        # Etiqueta al final derecho de la curva dentro del xlim
        p_vis = prec_arr[mask & (prec_arr >= xlim[0]) & (prec_arr <= xlim[1])]
        r_vis = rec_arr  [mask & (prec_arr >= xlim[0]) & (prec_arr <= xlim[1])]
        if len(p_vis) > 0:
            ax.text(p_vis[-1]-0.01, r_vis[-1]+0.015,
                    f"{ref['autor']}",
                    fontsize=7.5, color=GRIS_LIT, ha='right', style='italic')

    # ── Curvas iso-F1 de referencia propias (más claras) ──
    for f1v in [0.75, 0.80, 0.85, 0.90]:
        rec_arr = (f1v * prec_arr) / (2*prec_arr - f1v + 1e-9)
        mask = (rec_arr > 0.3) & (rec_arr <= 1.0)
        ax.plot(prec_arr[mask], rec_arr[mask],
                color='#E5E7EB', linewidth=1, linestyle='-', zorder=0)

    # ── Puntos de nuestros modelos ──
    for _, fila in df_pr.iterrows():
        col = COL_CL if fila['Tipo'] == 'Clásico' else COL_DL
        ax.scatter(fila['Precisión'], fila['Recall'],
                   c=col, s=fila['F1']*450, zorder=5,
                   edgecolors='white', linewidth=1.5, alpha=0.95)
        if xlim[0] <= fila['Precisión'] <= xlim[1] and ylim[0] <= fila['Recall'] <= ylim[1]:
            ax.annotate(fila['Modelo'],
                        (fila['Precisión'], fila['Recall']),
                        textcoords='offset points', xytext=(9,5),
                        fontsize=8.5, color='#374151')

    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_title(titulo, fontsize=10, color='#4B5563')
    ax.yaxis.grid(True, color='#E5E7EB', linewidth=0.8)
    ax.xaxis.grid(True, color='#E5E7EB', linewidth=0.8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

dibujar_panel(ax1, xlim=(0.0,1.05), ylim=(0.0,1.05),
              titulo='Vista completa  (escala 0–1)')
ax1.set_xlabel('Precisión  (fracción de alertas correctas)', fontsize=10)
ax1.set_ylabel('Recall  (fracción de deslizamientos detectados)', fontsize=10)

dibujar_panel(ax2, xlim=(0.66,0.88), ylim=(0.72,1.01),
              titulo='Zoom  (detalle de la región de interés)')
ax2.set_xlabel('Precisión  (fracción de alertas correctas)', fontsize=10)
ax2.set_ylabel('Recall  (fracción de deslizamientos detectados)', fontsize=10)

leyenda = [
    mpatches.Patch(color=COL_CL,   label='Modelo Clásico (este proyecto)'),
    mpatches.Patch(color=COL_DL,   label='Deep Learning (este proyecto)'),
    mpatches.Patch(color=GRIS_LIT, label='Curva ISO-F1 de literatura (autor, año)'),
]
fig.legend(handles=leyenda, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.04), framealpha=0.9)
fig.suptitle(
    'Trade-off Precisión vs Recall — Detección de Deslizamientos de Tierra\n'
    '(tamaño de burbuja proporcional al F1 Score · líneas grises = F1 reportado en literatura)',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_prec_recall.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Figura guardada: exploracion_prec_recall.png')


---
## 3. Exploración 2: Que canales espectrales discriminan mejor los deslizamientos?

**¿Pregunta?:** Existe diferencia estadistica entre la senal de pixeles de deslizamiento (positivos) vs. no-deslizamiento (negativos) en cada canal?

Calculamos el **delta** (diferencia de medias) como proxy de poder discriminativo.

In [ ]:
# ── Exploración 2: Poder discriminativo por canal espectral ───────────────────
# Colores por grupo de sensor:
#   RedEdge → rojo · Topografía/DEM → naranja · SAR → amarillo · Óptico → azul
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches

df_ch = pd.read_csv(f'{DATA_DIR}/channel_stats_by_class.csv').copy()
df_ch['|Delta|'] = df_ch['Delta'].abs()
df_ch = df_ch.sort_values('|Delta|', ascending=True).reset_index(drop=True)

COLORES_GRUPO = {
    'RedEdge'    : '#DC2626',   # rojo
    'Topografía' : '#F97316',   # naranja
    'SAR'        : '#EAB308',   # amarillo
    'Óptico'     : '#3B82F6',   # azul
}

def asignar_grupo(nombre):
    n = nombre.upper()
    if 'REDEDGE' in n or 'B6' in n or 'B7' in n: return 'RedEdge'
    if 'DEM' in n or 'SLOPE' in n or 'PENDIENTE' in n: return 'Topografía'
    if 'VV' in n or 'VH' in n or 'SAR' in n: return 'SAR'
    return 'Óptico'

df_ch['Grupo']  = df_ch['Nombre'].apply(asignar_grupo)
bar_cols = [COLORES_GRUPO[g] for g in df_ch['Grupo']]

fig, ax = plt.subplots(figsize=(11, 7))
ax.set_facecolor('white')

barras = ax.barh(df_ch['Nombre'], df_ch['|Delta|'],
                 color=bar_cols, edgecolor='white', linewidth=0.5)

for bar, val in zip(barras, df_ch['|Delta|']):
    ax.text(val + 0.006, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, color='#374151')

ax.set_xlabel(
    'Diferencia absoluta de intensidad media entre clases\n'
    '|µ(deslizamiento) − µ(sin deslizamiento)|  ·  escala 0 a 1',
    fontsize=10
)
ax.set_ylabel('Canal espectral', fontsize=11)
ax.set_title(
    'Poder Discriminativo por Canal Espectral\n'
    '(mayor valor = señal más diferente entre zona con y sin deslizamiento)',
    fontsize=12
)
ax.xaxis.grid(True, color='#E5E7EB', linewidth=0.8, zorder=0)
ax.yaxis.grid(False)

leyenda = [mpatches.Patch(color=c, label=g) for g, c in COLORES_GRUPO.items()]
ax.legend(handles=leyenda, fontsize=10, loc='lower right', framealpha=0.95,
          title='Grupo de sensor', title_fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_2_canales_delta.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Figura guardada: exploracion_2_canales_delta.png')


### Análisis Exploración 2

**Patrón claro:** Los canales **RedEdge3 (B7, |Delta|=0.807)** y **RedEdge2 (B6, |Delta|=0.563)** sobresalen notablemente sobre el resto. Estos canales de borde rojo de Sentinel-2 son altamente sensibles a la vegetacion y a suelos expuestos — exactamente lo que caracteriza una zona de deslizamiento fresco.

Los canales opticos convencionales (Azul, Verde, Rojo, NIR) muestran poca diferencia entre clases, lo que explica por que los modelos que priorizan esos canales tienen menor rendimiento.

In [ ]:
# ── Exploración 3: Brecha de señal entre clases — dot plot por grupo ──────────
# Opción A: colores por grupo (mismos que G3) + intensidad por clase.
#   Punto sólido/oscuro = con deslizamiento (señal alta)
#   Punto hueco/claro   = sin deslizamiento (señal baja)
# Forma varía por grupo para doble codificación visual.
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
import matplotlib.lines as mlines

df_ch = pd.read_csv(f'{DATA_DIR}/channel_stats_by_class.csv').copy()
df_ch['|Delta|'] = df_ch['Delta'].abs()

def asignar_grupo(nombre):
    n = nombre.upper()
    if 'REDEDGE' in n or 'B6' in n or 'B7' in n: return 'RedEdge'
    if 'DEM' in n or 'SLOPE' in n or 'PENDIENTE' in n: return 'Topografía'
    if 'VV' in n or 'VH' in n or 'SAR' in n: return 'SAR'
    return 'Óptico'

COLORES_GRUPO = {
    'RedEdge'    : '#DC2626',
    'Topografía' : '#F97316',
    'SAR'        : '#EAB308',
    'Óptico'     : '#3B82F6',
}
FORMAS_GRUPO = {
    'RedEdge'    : 'o',   # círculo
    'Topografía' : 's',   # cuadrado
    'SAR'        : '^',   # triángulo
    'Óptico'     : 'D',   # diamante
}

df_ch['Grupo'] = df_ch['Nombre'].apply(asignar_grupo)
df_top6 = df_ch.sort_values('|Delta|', ascending=False).head(6).reset_index(drop=True)
df_top6 = df_top6.sort_values('|Delta|', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_facecolor('white')

for i, (_, fila) in enumerate(df_top6.iterrows()):
    col    = COLORES_GRUPO[fila['Grupo']]
    forma  = FORMAS_GRUPO[fila['Grupo']]

    # Línea de conexión (gris)
    ax.plot([fila['Media_Neg'], fila['Media_Pos']], [i, i],
            color='#D1D5DB', linewidth=2.5, zorder=1, solid_capstyle='round')

    # Sin deslizamiento: punto hueco (facecolor blanco, borde de color)
    ax.scatter(fila['Media_Neg'], i, marker=forma, s=100,
               facecolors='white', edgecolors=col, linewidth=2, zorder=3)

    # Con deslizamiento: punto sólido (color intenso)
    ax.scatter(fila['Media_Pos'], i, marker=forma, s=100,
               facecolors=col, edgecolors='white', linewidth=1, zorder=4)

    # Delta encima
    mid_x = (fila['Media_Neg'] + fila['Media_Pos']) / 2
    ax.text(mid_x, i+0.28, f'Δ={fila["|Delta|"]:.3f}',
            ha='center', va='bottom', fontsize=8.5, color='#6B7280')

ax.set_yticks(range(len(df_top6)))
ax.set_yticklabels(df_top6['Nombre'], fontsize=10)
ax.set_xlabel('Intensidad normalizada del canal  (escala 0–1)', fontsize=10)
ax.set_ylabel('Canal espectral', fontsize=10)
ax.set_title(
    'Brecha de Señal entre Clases — Top 6 Canales Más Discriminativos\n'
    '(punto sólido = con deslizamiento · punto hueco = sin deslizamiento · Δ = brecha)',
    fontsize=11
)
ax.set_xlim(0, None)   # eje desde 0
ax.xaxis.grid(True, color='#E5E7EB', linewidth=0.8)
ax.yaxis.grid(False)

# Leyenda: grupos de sensor (color + forma)
handles = [
    mlines.Line2D([0],[0], marker=FORMAS_GRUPO[g], color='w',
                  markerfacecolor=COLORES_GRUPO[g], markeredgecolor=COLORES_GRUPO[g],
                  markersize=9, label=g)
    for g in COLORES_GRUPO
]
handles += [
    mlines.Line2D([0],[0], marker='o', color='w', markerfacecolor='#374151',
                  markersize=9, label='Con deslizamiento (sólido)'),
    mlines.Line2D([0],[0], marker='o', color='w', markerfacecolor='white',
                  markeredgecolor='#374151', markeredgewidth=2,
                  markersize=9, label='Sin deslizamiento (hueco)'),
]
ax.legend(handles=handles, fontsize=8.5, loc='lower right',
          framealpha=0.95, ncol=2)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_3_medias_clase.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Figura guardada: exploracion_3_medias_clase.png')


In [ ]:
# ── Exploración 4: Ranking F1 y Recall ───────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel izquierdo: F1 ──
df_f1 = df_models.sort_values('F1 medio', ascending=True).reset_index(drop=True)
colores_rank = ['#DC2626' if m == 'Random Forest'
                else ('#2CA02C' if t == 'Clásico' else '#7C3AED')
                for m, t in zip(df_f1['Modelo'], df_f1['Tipo'])]

barras_izq = axes[0].barh(df_f1['Modelo'], df_f1['F1 medio'],
                           color=colores_rank, alpha=0.88, edgecolor='white')
for bar, val in zip(barras_izq, df_f1['F1 medio']):
    axes[0].text(val+0.01, bar.get_y()+bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

axes[0].axvline(0.80, color='#DC2626', linewidth=1.4, linestyle='--')   # ROJO
axes[0].text(0.803, 0.25, 'Umbral\noperativo\nF1=0.80',
             fontsize=8, color='#DC2626', va='bottom')
axes[0].set_xlabel('F1 Score  (media harmónica de Precisión y Recall; 0–1)', fontsize=10)
axes[0].set_ylabel('Modelo de Machine Learning', fontsize=10)
axes[0].set_title('Ranking por F1 Score\n(rojo = mejor modelo)')
axes[0].set_xlim(0, 1.0)
axes[0].xaxis.grid(True, color='#E5E7EB', linewidth=0.8)
axes[0].yaxis.grid(False)
axes[0].set_facecolor('white')

# ── Panel derecho: Recall ──
df_rec = df_models.dropna(subset=['Recall']).sort_values('Recall', ascending=True).reset_index(drop=True)
colores_rec = ['#DC2626' if m == 'Random Forest' else '#9CA3AF' for m in df_rec['Modelo']]

barras_der = axes[1].barh(df_rec['Modelo'], df_rec['Recall'],
                           color=colores_rec, alpha=0.88, edgecolor='white')
for bar, val in zip(barras_der, df_rec['Recall']):
    axes[1].text(val+0.01, bar.get_y()+bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

axes[1].axvline(0.90, color='#DC2626', linewidth=1.4, linestyle='--')
axes[1].text(0.913, 0.2,
             'Recall mínimo\npara sistema\nde alerta (0.90)',
             fontsize=8, color='#DC2626', va='bottom')
axes[1].set_xlabel('Recall  (fracción de deslizamientos reales detectados; 0–1)', fontsize=10)
axes[1].set_ylabel('Modelo de Machine Learning', fontsize=10)
axes[1].set_title('Recall por Modelo\n(crítico para alertas tempranas)')
axes[1].set_xlim(0, 1.05)
axes[1].xaxis.grid(True, color='#E5E7EB', linewidth=0.8)
axes[1].yaxis.grid(False)
axes[1].set_facecolor('white')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_4_recall.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Figura guardada: exploracion_4_recall.png')


---
## 4. Exploración 5: Variabilidad entre folds — consistencia del modelo

**¿Pregunta?:** Las métricas promedio ocultan variabilidad. Un modelo con F1=0.83 pero std=0.03 puede ser poco fiable operativamente. ¿Son los modelos estables entre particiones del dataset?

Cargamos los resultados por fold de cada modelo para comparar su distribucion real.

In [ ]:
# ── Exploración 5: Cargar resultados por fold de cada modelo ──────────────────
# Los JSONs de folds estan en visualizacion_datos/data/folds/ (incluidos en el repo)
import json as _json

FOLDS_DIR = f'{DATA_DIR}/folds'

def load_folds_classical(path):
    with open(path) as f:
        d = _json.load(f)
    return [fold['best_f1'] for fold in d['folds']]

def load_folds_dl(path, key='f1_pixel_thr05'):
    with open(path) as f:
        d = _json.load(f)
    return [fold[key] for fold in d['folds']]

fold_data = {}
try:
    fold_data['Logistic Reg.']  = load_folds_classical(f'{FOLDS_DIR}/logistic_regression_folds.json')
    fold_data['SVM (RBF)']      = load_folds_classical(f'{FOLDS_DIR}/svm_folds.json')
    fold_data['Random Forest']  = load_folds_classical(f'{FOLDS_DIR}/random_forest_folds.json')
    fold_data['ResNet-50']      = load_folds_dl(f'{FOLDS_DIR}/resnet50_folds.json', key='f1_thr05')
    fold_data['U-Net ResNet34'] = load_folds_dl(f'{FOLDS_DIR}/unet_folds.json', key='f1_pixel_thr05')
    # EfficientNet-B4: sin datos por fold, se omite
    print('Folds cargados OK:')
    for k, v in fold_data.items():
        print(f'  {k:<20} folds={len(v)}  mean={np.mean(v):.4f}  std={np.std(v):.4f}')
except FileNotFoundError as e:
    print(f'Archivo no encontrado: {e}')
    fold_data = {}


In [ ]:
# ── Exploración 5a: Boxplot de F1 por modelo (distribucion real 5 folds) ──────
if fold_data:
    order = sorted(fold_data.keys(), key=lambda k: np.median(fold_data[k]), reverse=True)
    data_ordered = [fold_data[k] for k in order]

    VERDE   = '#2ca02c'
    PURPURA = '#9467bd'
    dl_models = {'ResNet-50', 'U-Net ResNet34'}
    box_colors = [PURPURA if m in dl_models else VERDE for m in order]

    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot(data_ordered, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))

    for patch, color in zip(bp['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    for i, vals in enumerate(data_ordered, 1):
        ax.scatter([i]*len(vals), vals, color='black', s=30, zorder=5, alpha=0.7)

    ax.set_xticks(range(1, len(order)+1))
    ax.set_xticklabels(order, rotation=15, ha='right')
    ax.set_ylabel('F1 Score (por fold)')
    ax.set_title('Distribucion de F1 Score por Modelo — 5-fold CV\n(cada punto = un fold, caja = rango intercuartil)')
    ax.set_ylim(0.35, 1.0)
    ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5)
    ax.text(len(order) + 0.1, 0.803, 'F1 = 0.80', fontsize=9, color='gray', va='bottom')

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=VERDE, label='Clasico'),
                       Patch(facecolor=PURPURA, label='Deep Learning')]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/exploracion_5a_boxplot_folds.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada: exploracion_5a_boxplot_folds.png')

### Análisis Exploración 5a — Boxplot

**Patrón de estabilidad:** Random Forest muestra la caja mas estrecha — sus 5 folds oscilan apenas 0.012 puntos F1 (0.824–0.848). En cambio, SVM (RBF) presenta la mayor dispersion (std=0.030), con el fold 1 cayendo hasta F1=0.753, por debajo del umbral operativo de 0.80.

**U-Net ResNet-34** presenta una distribucion compacta pero en un rango bajo (0.68–0.71), lo que indica que su bajo rendimiento es sistematico y no un artefacto de una mala particion.

In [ ]:
# ── Exploración 5b: Heatmap fold x modelo ─────────────────────────────────────
if fold_data:
    df_folds = pd.DataFrame(fold_data, index=[f'Fold {i}' for i in range(1, 6)])

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.heatmap(df_folds.T, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0.40, vmax=0.90, linewidths=0.5, linecolor='white',
                ax=ax, cbar_kws={'label': 'F1 Score'})

    ax.set_title('Heatmap F1 Score — Fold x Modelo\n(verde = mejor rendimiento, rojo = peor)')
    ax.set_xlabel('Particion (fold)')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/exploracion_5b_heatmap_folds.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada: exploracion_5b_heatmap_folds.png')

    fold_means = df_folds.mean(axis=1)
    worst_fold = fold_means.idxmin()
    print(f'\nFold con menor F1 promedio entre modelos: {worst_fold} (media={fold_means.min():.4f})')
    print('Medias por fold:')
    print(fold_means.round(4).to_string())

### Análisis Exploración 5b — Heatmap

**Pregunta clave:** Si hay un fold sistemáticamente mas difícil para todos los modelos, eso sugiere una particion de datos geográficamente sesgada — ese fold contiene imagenes con caracteristicas espectrales distintas al resto.

La columna con valores mas bajos (mas rojo) identifica la particion mas exigente del dataset. Este analisis complementa la evaluacion LORO (Leave-One-Region-Out) del dashboard interactivo, donde el sesgo geografico se hace aun mas evidente.

---
## 5. El Hallazgo

Despues del analisis exploratorio, se articulan dos hallazgos principales:

---

### Hallazgo 1: Random Forest supera a las redes neuronales profundas

**Anomalía encontrada:** Contra la intuicion dominante en computer vision, el modelo **Random Forest (F1=0.837)** supera a todos los modelos de Deep Learning. Esto se explica porque:
- El dataset Landslide4Sense (3799 muestras) es **relativamente pequeño** para entrenar redes profundas desde cero
- Las features engineered (14 canales ya pre-procesados) dan ventaja a los metodos de ensamble sobre las CNN que deben aprender representaciones
- U-Net (F1=0.444) sufre especialmente por su alta capacidad parametrica sin suficientes datos de entrenamiento

**Implicación de negocio:** Para sistemas de deteccion con datasets limitados y features bien definidas, los modelos clásicos de ML son la primera eleccion, no las arquitecturas profundas.

---

### Hallazgo 2: Los canales RedEdge (B6, B7) son la senal mas discriminativa

**Correlación descubierta:** Las bandas espectrales de **borde rojo** (RedEdge) de Sentinel-2 muestran una diferencia de medias (Delta) hasta **10 veces mayor** que los canales opticos convencionales. Esta senal corresponde fisicamente a la respuesta espectral del suelo desnudo expuesto durante un deslizamiento.

**Implicación de negocio:** Priorizar imagenes con bandas RedEdge disponibles (Sentinel-2 Nivel 2A) maximizara la precision de deteccion. El **DEM de elevacion (Delta=0.195)** y **SAR-VH (Delta=0.188)** complementan la senal optica.

---

**Estos hallazgos guiaran el diseno del Dashboard aclaratorio (Fase 2).**

In [ ]:
# ── Resumen cuantitativo del hallazgo ─────────────────────────────────────────
print('='*60)
print('RESUMEN DE HALLAZGOS — EXPLORACION')
print('='*60)
print()
print('Hallazgo 1: Ranking de modelos por F1 Score')
for _, row in df_models.sort_values('F1 medio', ascending=False).iterrows():
    marker = '>>' if row['Modelo'] == 'Random Forest' else '  '
    print(f'  {marker} {row["Modelo"]:<25} F1={row["F1 medio"]:.4f}  [{row["Tipo"]}]')

print()
print('Hallazgo 2: Top 5 canales mas discriminativos (|Delta media|)')
df_channels_sorted = df_channels.copy()
df_channels_sorted['|Delta|'] = df_channels_sorted['Delta'].abs()
for _, row in df_channels_sorted.sort_values('|Delta|', ascending=False).head(5).iterrows():
    print(f'  Canal {int(row["Canal"]):2d} — {row["Nombre"]:<25} |Delta|={row["|Delta|"]:.4f}')

print()
if fold_data:
    print('Hallazgo 3: Estabilidad entre folds (std F1)')
    for modelo, folds in sorted(fold_data.items(), key=lambda x: np.std(x[1])):
        marker = '>>' if modelo == 'Random Forest' else '  '
        print(f'  {marker} {modelo:<20} mean={np.mean(folds):.4f}  std={np.std(folds):.4f}  [{"ESTABLE" if np.std(folds) < 0.015 else "VARIABLE"}'  + ']')

print()
print('-> Estos hallazgos seran el centro del Dashboard aclaratorio.')
print('='*60)
